# Build a model RDM

Representational dissimilarity matrices compare examples by the geometry of a model representation. This notebook uses `tl.bridge.rsatoolbox` when available, and falls back to `tl.export.xarray` otherwise.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import torch
import torchlens as tl

shared_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "notebooks" / "total_audit" / "_shared.py"
    if candidate.exists():
        shared_path = candidate
        break
if shared_path is None:
    raise FileNotFoundError("Could not find notebooks/total_audit/_shared.py")

spec = importlib.util.spec_from_file_location("torchlens_total_audit_shared", shared_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {shared_path}")
shared = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = shared
spec.loader.exec_module(shared)
tiny_model = shared.tiny_model

torch.manual_seed(0)
model = tiny_model(seed=7).eval()
x = torch.randn(2, 4)
x = torch.randn(5, 4)

The rsatoolbox route is five lines: capture, convert, compute distances, wrap as an RDM, and inspect the matrix.

In [ ]:
log = tl.log_forward_pass(model, x, layers_to_save=["linear_2_3"])
try:
    from scipy.spatial.distance import pdist, squareform

    dataset = tl.bridge.rsatoolbox.dataset(log)
    rdm = squareform(pdist(dataset.measurements, metric="correlation"))
    print("backend: rsatoolbox")
except ImportError:
    assembly = tl.export.xarray(log)
    relu_units = assembly.sel(neuroid=assembly.layer == "linear_2_3")
    rdm = (
        (relu_units - relu_units.mean("neuroid")) @ (relu_units - relu_units.mean("neuroid")).T
    ).values
    print("backend: xarray fallback; install torchlens[neuro] to complete the rsatoolbox recipe")
rdm.round(3)

Rows and columns correspond to the five input presentations. Smaller values mean more similar representations under the chosen distance.